# 03 — Robot Baseline Strategy
Indicators, score generation and corrected portfolio backtest.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime, add_robot_scores
from src.backtest import run_portfolio_backtest
from src.metrics import portfolio_metrics, yearly_performance, monthly_performance


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "bist100_robot_clean.parquet"
)
market_prices = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)

market_regime = build_market_regime(market_features)
scored_prices = add_robot_scores(
    stock_features,
    market_regime,
    StrategyConfig(),
)


In [ ]:
equity_df, trades_df = run_portfolio_backtest(
    scored_prices,
    StrategyConfig(),
    PortfolioConfig(),
)

metrics = portfolio_metrics(equity_df, trades_df)
display(pd.DataFrame([metrics]))
display(trades_df.head())


In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

equity_df.to_parquet(results_dir / "baseline_equity.parquet", index=False)
trades_df.to_csv(results_dir / "baseline_trades.csv", index=False)
yearly_performance(equity_df).to_csv(
    results_dir / "baseline_yearly.csv", index=False
)
monthly_performance(equity_df).to_csv(
    results_dir / "baseline_monthly.csv", index=False
)
